In [1]:
# ---- STARTUP CELL ----
from dotenv import load_dotenv
import os
import sys

# Hardcoded absolute path — same fix that worked in Week 1 and 2
load_dotenv(dotenv_path='/Users/munizasiddiqui/easyf1/.env')
API_KEY = os.getenv("GEMINI_API_KEY")

# Add project root to path
sys.path.append('/Users/munizasiddiqui/easyf1')

print(f"API Key loaded: {'YES ✅' if API_KEY else 'NO ❌'}")
print(f"Working directory: {os.getcwd()}")



API Key loaded: YES ✅
Working directory: /Users/munizasiddiqui/easyf1/notebooks


In [2]:
#Importing libraries and setting up Gemini for building RAG pipeline 
#Setting connection to Gemini API

from google import genai 
from google.genai import types 
import chromadb
import pandas as pd 
import json
import os

#setting up Gemini client by new google genai libary 
client = genai.Client(api_key = API_KEY)

print('Libraries imported')
print("Gemini client ready")

Libraries imported
Gemini client ready


In [3]:
#Updated embedding function
#usinf fastembed, no api calls, runs locally, no rate limits

from fastembed import TextEmbedding

#loading the model once so it stays in the memory 
embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

def embed_text(text):
    """
    converts text into a vector using fastembed
    locally run so no api calls or rate limits 

    text:str
        The text to embed 

    Returns: 
    embedding : list of floats
        384-dimensional vector representing text meaning

    """
    embedding = list(embedding_model.embed([text]))[0]
    return embedding.tolist()

test = embed_text("Hamilton pitted under safety car at silverstone")
print(f"embed_text function ready")
print(f"Embedding Lenght:{len(test)} dimensions")
 

/Users/munizasiddiqui/easyf1/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


embed_text function ready
Embedding Lenght:384 dimensions


In [4]:
#Loading chunks + ChromaDB
#Loading the saved chunks from week 2
chunks_path = '/Users/munizasiddiqui/easyf1/data/2023_british_gp_chunks.csv'
chunks_df = pd.read_csv(chunks_path)

print(f"Loaded {len(chunks_df)} chunks from CSV")
print(f"\nColumns : {list(chunks_df.columns)}")

#Setting up ChromaDB, has persistant storage so data survives restarts
chroma_path = '/Users/munizasiddiqui/easyf1/chromadb'
chroma_client = chromadb.PersistentClient(path = chroma_path)

collection = chroma_client.get_collection(name="easyf1_races")
print(f"Collection loaded ✅")
print(f"Chunks in collection: {collection.count()}")
print("ChromaDB collection 'easyf1_races' created")

Loaded 971 chunks from CSV

Columns : ['text', 'race', 'circuit', 'year', 'lap', 'driver', 'team', 'compound', 'tyre_age', 'position', 'sc_active', 'vsc_active', 'had_pit']
Collection loaded ✅
Chunks in collection: 971
ChromaDB collection 'easyf1_races' created


In [5]:
#Updated cell 5 
# Re-Embed all 971 chunks 
import time 

#Step 1 --> fresh collection
chroma_client.delete_collection("easyf1_races")
collection = chroma_client.create_collection(
    name = 'easyf1_races',
    metadata = {"hnsw:space": "cosine"}
)

print(f"fresh collection created")

#Step 2 --> Embed and store all chunks 
batch_size = 100
total = len(chunks_df)
success = 0

print(f"Embedding {total} chunks locally - no rate limits")
print()

for i in range (0, total, batch_size):
    batch = chunks_df.iloc[i:i + batch_size]
    texts = batch ['text'].tolist()
    ids = [f"chunk_{j}"for j in range(i, i + len(batch))]
    metadatas = []

    for _, row in batch.iterrows():
        metadatas.append({
            'race' :      str(row['race']),
            'circuit' :   str(row['circuit']),
            'year' :      int(row['year']),
            'driver' :    str(row['driver']),
            'lap' :       int(row['lap']),
            'team' :      str(row['team']),
            'compound' :  str(row['compound']),
            'tyre_age' :  int(row['tyre_age']),
            'position' :  int(row['position']),
            'sc_active' : bool(row['sc_active']),
            'vsc_active' :bool(row['vsc_active']),
            'had_pit' :   bool(row['had_pit'])
        })

    #embedding entire batch at once
    embeddings = [embed_text(text) for text in texts]

    collection.add(
        documents = texts,
        embeddings = embeddings,
        metadatas = metadatas,
        ids = ids 
    )

    success += len(batch)
    print(f"Progress: {success}/{total} chunks embedded...")

print()
print(f"All done")
print(f"Total in chromaDB {collection.count()}")
    

fresh collection created
Embedding 971 chunks locally - no rate limits

Progress: 100/971 chunks embedded...
Progress: 200/971 chunks embedded...
Progress: 300/971 chunks embedded...
Progress: 400/971 chunks embedded...
Progress: 500/971 chunks embedded...
Progress: 600/971 chunks embedded...
Progress: 700/971 chunks embedded...
Progress: 800/971 chunks embedded...
Progress: 900/971 chunks embedded...
Progress: 971/971 chunks embedded...

All done
Total in chromaDB 971


In [6]:
#Querying the RAG knowledge base 
#finding the relevant chunk for a given question using semantic similarity 
#Converting question to vector then ChromaDB finds the most closest chunk vectors

def query_easyf1 (question, n_results = 5, filters = None):
    """
    Searches ChromaDB for chunk relavant the question 
    using sementic similarity, finding meaning, not exact words 

    Parameters:

    question: str
        the strategy question to search for
    n_results: int
        How many chunks to recieve, default (5) 
    filter: dict, optional 
        chormaDB where clause for metadata filtering 
        eg: {"$and: [{'driver' : {"eq": "HAM"}},
                    {"lap": {"$gt": 26}}]}

    returns()

    results: dict
        contains documents, distances and metadatas
    """

    #Step 1, embed the questio. with same model
    #We  must use the exact same model we used to embed the chunks, or the vectors are incompatible
    question_embedding = embed_text(question)

    query_params = {
        "query_embeddings" : [question_embedding],
        "n_results" : n_results,
        "include" : ['documents', 'metadatas', 'distances']
    }

    if filters:
        query_params ['where'] = filters

    #Step 2, searching chromaDb for similar chunks
    results = collection.query(**query_params)
    return results 

#Testing it with a scenario
question = "Hamilton on old medium tyres safety car deployed silverstone what should Mercedes do?"

print(f"Questions: {question}")
print()
print("Most relevant chunks retrieved:")

results = query_easyf1(question)

for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    
    print(f"\nchunk {i+1} - Similarity: {1 - distance: .2%}")
    print(f"\Dirver: {metadata['driver']} | Lap: {metadata['lap']} | Compound: {metadata['compound']}")
    print(f"Sc acitve: {metadata['sc_active']} | Had Pit: {metadata['had_pit']}")
    print(f"Text: {doc[:150]}...")
    


Questions: Hamilton on old medium tyres safety car deployed silverstone what should Mercedes do?

Most relevant chunks retrieved:

chunk 1 - Similarity:  78.19%
\Dirver: HAM | Lap: 35 | Compound: SOFT
Sc acitve: True | Had Pit: False
Text: Race 2023 British Grand Prix at Silverstone. Lap 35 of 52. Driver: Lewis Hamilton (HAM) racing for Mercedes. Running in position 3. Tyre Compound: SOF...

chunk 2 - Similarity:  77.92%
\Dirver: HAM | Lap: 33 | Compound: MEDIUM
Sc acitve: True | Had Pit: True
Text: Race 2023 British Grand Prix at Silverstone. Lap 33 of 52. Driver: Lewis Hamilton (HAM) racing for Mercedes. Running in position 3. Tyre Compound: MED...

chunk 3 - Similarity:  77.89%
\Dirver: HAM | Lap: 36 | Compound: SOFT
Sc acitve: True | Had Pit: False
Text: Race 2023 British Grand Prix at Silverstone. Lap 36 of 52. Driver: Lewis Hamilton (HAM) racing for Mercedes. Running in position 3. Tyre Compound: SOF...

chunk 4 - Similarity:  77.79%
\Dirver: HAM | Lap: 38 | Compound: SOFT
Sc aci

In [11]:
# ---- DRIVER DETECTION FOR AUTO-FILTERING ----
# Maps full names and common references to driver codes
# Used to automatically add metadata filters when a driver is mentioned

DRIVER_LOOKUP = {
    'hamilton'    : 'HAM',
    'lewis'       : 'HAM',
    'verstappen'  : 'VER',
    'max'         : 'VER',
    'leclerc'     : 'LEC',
    'charles'     : 'LEC',
    'perez'       : 'PER',
    'checo'       : 'PER',
    'russell'     : 'RUS',
    'george'      : 'RUS',
    'sainz'       : 'SAI',
    'carlos'      : 'SAI',
    'norris'      : 'NOR',
    'lando'       : 'NOR',
    'alonso'      : 'ALO',
    'fernando'    : 'ALO',
    'gasly'       : 'GAS',
    'pierre'      : 'GAS',
    'ocon'        : 'OCO',
    'esteban'     : 'OCO',
    'stroll'      : 'STR',
    'lance'       : 'STR',
    'bottas'      : 'BOT',
    'valtteri'    : 'BOT',
    'zhou'        : 'ZHO',
    'magnussen'   : 'MAG',
    'kevin'       : 'MAG',
    'hulkenberg'  : 'HUL',
    'nico'        : 'HUL',
    'tsunoda'     : 'TSU',
    'yuki'        : 'TSU',
    'albon'       : 'ALB',
    'alex'        : 'ALB',
    'sargeant'    : 'SAR',
    'logan'       : 'SAR',
    'piastri'     : 'PIA',
    'oscar'       : 'PIA',
}

def detect_driver(question):
    """
    Detects if a specific driver is mentioned in the question.
    Returns their 3-letter code or None if no driver detected.
    
    Parameters:
    -----------
    question : str
        The user's question
    
    Returns:
    --------
    driver_code : str or None
        e.g. 'HAM' if Hamilton mentioned, None if no driver found
    """
    question_lower = question.lower()
    
    for name, code in DRIVER_LOOKUP.items():
        if name in question_lower:
            return code
    
    return None

def detect_lap_range(question, total_laps=52):
    """
    Detects if question refers to a specific part of the race.
    Returns a lap filter or None.
    
    Parameters:
    -----------
    question : str
        The user's question
    total_laps : int
        Total laps in the race (default 52 for British GP)
    
    Returns:
    --------
    lap_filter : dict or None
        ChromaDB lap filter or None if no range detected
    """
    question_lower = question.lower()
    half           = total_laps // 2
    
    if 'second half' in question_lower or 'late race' in question_lower:
        return {"lap": {"$gt": half}}
    
    elif 'first half' in question_lower or 'early race' in question_lower:
        return {"lap": {"$lt": half}}
    
    elif 'final laps' in question_lower or 'last laps' in question_lower:
        return {"lap": {"$gt": total_laps - 10}}
    
    elif 'opening laps' in question_lower or 'first laps' in question_lower:
        return {"lap": {"$lt": 10}}
    
    return None

# Test it
print(detect_lap_range("What happened in the second half?"))
print(detect_lap_range("What tyre in the first half?"))
print(detect_lap_range("What happened in the final laps?"))
print(detect_lap_range("Hamilton pitted under safety car"))

{'lap': {'$gt': 26}}
{'lap': {'$lt': 26}}
{'lap': {'$gt': 42}}
None


In [14]:
# ---- NUCLEAR RESET ----
# Delete everything and start fresh
chroma_client.delete_collection("easyf1_races")

# Recreate clean collection
collection = chroma_client.create_collection(
    name="easyf1_races",
    metadata={"hnsw:space": "cosine"}
)

print(f"Clean collection created ✅")
print(f"Count: {collection.count()}")

Clean collection created ✅
Count: 0


In [17]:
#Testing fastembed 
from fastembed import TextEmbedding

#Downloading the model on first run
embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

#Testing with a simple F1 sentence
test_text = "Hamilton pitted under safety car on lap 33 at Silverstone"
embedding = list(embedding_model.embed([test_text]))[0]

print(f"Embedding created")
print(f"Length: {len(embedding)} dimensions")
print(f"First five values: {list(embedding[:5])}")

Fetching 5 files: 100%|██████████| 5/5 [00:08<00:00,  1.70s/it]


Embedding created
Length: 384 dimensions
First five values: [0.01195516, 0.00852209, -0.054148257, 0.010319402, -0.05126717]


In [12]:
#Now connecting to RAG to Gemini
#Building the complete rag pipeline

def ask_easyf1(question, filters = None):
    """
    RAG Pipeline
    retrieves relevant chunk for the ChromaDB
    Passes them to Gemini as context
    Gimini reason over real data, not memory 


    Parameters:
    question: str
        The quetion about the strategy from the user

    answer: str
        Response from Gemini in real F1 data 


    """
    # Step 1: Auto-detect driver and lap range, build combined filter
    if filters is None:
        detected_driver    = detect_driver(question)
        detected_lap_range = detect_lap_range(question)
        
        conditions = []
        
        if detected_driver:
            conditions.append({"driver": {"$eq": detected_driver}})
            print(f"Auto-filter: {detected_driver} detected")
        
        if detected_lap_range:
            conditions.append(detected_lap_range)
            print(f"Auto-filter: lap range detected")
        
        if len(conditions) == 1:
            filters = conditions[0]
        elif len(conditions) == 2:
            filters = {"$and": conditions}



    #Step 1 is to retrieve relevant chunks
    results = query_easyf1(question, filters = filters)
    chunks = results ['documents'][0]
    metadatas = results ['metadatas'][0]

    #Step 2 is formatting retrieved chunks into context string
    #This is what gets passed to gemini alongside the question

    context = "\n\n".join([
        f"[Historical data {i+1}]\n {chunk}"
        for i, chunk in enumerate (chunks)
    ])

    #Step 3, detect question type
    #Strategy keywords suggest a decision is needed
    #For Analytical keywords suggests a factual answer is required
    strategy_keywords = ['should', 'pit now', 'recommend', 'what should', 
                         'do we pit', 'strategy for']
    
    analytical_keywords = ['was', 'did', 'how long', 'which lap', 'what happened', 'successful',
                           'when did', 'how many', 'what tyre', 'final position']
    
    question_lower = question.lower()
    is_strategy = any(kw in question_lower for kw in strategy_keywords)
    is_analytical = any(kw in question_lower for kw in analytical_keywords)

    #Step 4 is to build prompt with context injected
    #Telling Gemini to use the data and not it's memory
    if is_strategy and not is_analytical:
        prompt = f"""You are EasyF1, an AI race strategy assistant
You have been given Real F1 race data retrieved from a database.
Use only this data to make recommendation. Do not use general knowledge. 

Real Race Data:
{context}

Question: {question}

Respond exactly in this format:
VERDICT: [PIT NOW/ STAY OUT/ PIT IN N LAPS]
CONFIFDENCE: [LOW/ MEDIUM/ HIGH]
RESONING: [2-3 sentences using specific numbers from the data above]
RISK: [1 sentence on the biggest risk]
HISTORICAL PARALLEL: [Reference the specific race data provided]
"""
    else:
        #analytical format
        prompt = f"""You are EasyF1, an AI race strategy assistant
You have been given Real F1 race data retrieved from a database.
Use only this data to make recommendation. Do not use general knowledge.
if the data does not contain enough information to answer,say so clearly

Real Race Data:
{context}

Question: {question}

Provide a clear, concise answer in 2-4 sentences using specific 
facts and numbers from the data above.
"""

    #step 5: Send to Gemini
    # response = client.models.generate_content(
    #     model = "models/gemini-2.5-flash-lite",
    #     contents = prompt
    # )

    # return response.text
    import time
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="models/gemini-2.5-flash-lite",
                contents=prompt
            )
            return response.text
        except Exception as e:
            if '503' in str(e) or '429' in str(e):
                print(f"Server busy — waiting 30 seconds (attempt {attempt+1}/3)...")
                time.sleep(30)
            else:
                raise e
    return "Error: Could not get response after 3 attempts"

#Testing the full RAG Pipeline 
print(f"testing full RAG Pipeline")

question = "Hamilton on old medium tyres safety car deployed silverstone what should Mercedes do?"
answer = ask_easyf1(question)

print(f"question: {question}")
print(f"Easy F1 Answer:")
print(answer)

testing full RAG Pipeline
Auto-filter: HAM detected
Server busy — waiting 30 seconds (attempt 1/3)...
Server busy — waiting 30 seconds (attempt 2/3)...
Server busy — waiting 30 seconds (attempt 3/3)...
question: Hamilton on old medium tyres safety car deployed silverstone what should Mercedes do?
Easy F1 Answer:
Error: Could not get response after 3 attempts


In [29]:
question1 = "What tyre compound did Hamilton use in the second half of the race?"
answer1   = ask_easyf1(question1)
print("QUESTION 1:", question1)
print(answer1)
print("="*60)

QUESTION 1: What tyre compound did Hamilton use in the second half of the race?
VERDICT: STAY OUT
CONFIDENCE: LOW
RESONING: The provided data only covers the first half of the race, from lap 2 to lap 17, and all instances show Lewis Hamilton on MEDIUM tyres. There is no data available for the second half of the race to determine his tyre compound.
RISK: The risk is that we do not have any data to support a recommendation for the second half of the race.
HISTORICAL PARALLEL: Historical data 1, 2, 3, 4, and 5 all show Lewis Hamilton on MEDIUM tyres during the first half of the 2023 British Grand Prix.


In [16]:
# ---- DEBUG: WHAT DID RAG ACTUALLY RETRIEVE? ----
question1 = "What tyre compound did Hamilton use in the second half of the race?"

results = query_easyf1(question1)

print("Chunks retrieved for this question:")
print("="*60)

for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"\nChunk {i+1} — Similarity: {1-distance:.2%}")
    print(f"Driver: {metadata['driver']} | Lap: {metadata['lap']}")
    print(f"Text: {doc[:150]}...")

Chunks retrieved for this question:

Chunk 1 — Similarity: 72.39%
Driver: GAS | Lap: 32
Text: Race 2023 British Grand Prix at Silverstone. Lap 32 of 52. Driver: GAS (Alpine). Running in position 11. Tyre Compound: SOFT, tyre_age: 4 laps. Stint ...

Chunk 2 — Similarity: 72.25%
Driver: GAS | Lap: 2
Text: Race 2023 British Grand Prix at Silverstone. Lap 2 of 52. Driver: GAS (Alpine). Running in position 9. Tyre Compound: MEDIUM, tyre_age: 2 laps. Stint ...

Chunk 3 — Similarity: 72.13%
Driver: DEV | Lap: 2
Text: Race 2023 British Grand Prix at Silverstone. Lap 2 of 52. Driver: DEV (AlphaTauri). Running in position 17. Tyre Compound: SOFT, tyre_age: 2 laps. Sti...

Chunk 4 — Similarity: 72.03%
Driver: OCO | Lap: 2
Text: Race 2023 British Grand Prix at Silverstone. Lap 2 of 52. Driver: OCO (Alpine). Running in position 12. Tyre Compound: SOFT, tyre_age: 2 laps. Stint n...

Chunk 5 — Similarity: 71.77%
Driver: GAS | Lap: 37
Text: Race 2023 British Grand Prix at Silverstone. Lap 37 of 52. Dr

In [17]:
# ---- TEST: DOES "HAM" VS "HAMILTON" MATTER? ----

q1 = "What tyre compound did HAM use in the second half of the race?"
q2 = "What tyre compound did Hamilton use in the second half of the race?"

results1 = query_easyf1(q1)
results2 = query_easyf1(q2)

print("Using 'HAM':")
for m in results1['metadatas'][0]:
    print(f"  {m['driver']} - Lap {m['lap']}")

print("\nUsing 'Hamilton':")
for m in results2['metadatas'][0]:
    print(f"  {m['driver']} - Lap {m['lap']}")

Using 'HAM':
  HAM - Lap 2
  HAM - Lap 15
  HAM - Lap 17
  HAM - Lap 18
  HAM - Lap 5

Using 'Hamilton':
  GAS - Lap 32
  GAS - Lap 2
  DEV - Lap 2
  OCO - Lap 2
  GAS - Lap 37


In [24]:
# ---- RE-TEST THE FAILING QUESTION ----
q2 = "What tyre compound did Hamilton use in the second half of the race?"
results2 = query_easyf1(q2)

print("Using 'Hamilton' (full name) after fix:")
for m in results2['metadatas'][0]:
    print(f"  {m['driver']} - Lap {m['lap']}")

Using 'Hamilton' (full name) after fix:
  HAM - Lap 2
  HAM - Lap 14
  HAM - Lap 15
  HAM - Lap 17
  HAM - Lap 11


In [25]:
# ---- PURE RETRIEVAL TEST (NO GEMINI CALL) ----
results2 = collection.query(
    query_embeddings=[embed_text("What tyre compound did Hamilton use in the second half of the race?")],
    n_results=5
)

print("Retrieved drivers and laps:")
for m in results2['metadatas'][0]:
    print(f"  {m['driver']} - Lap {m['lap']}")

Retrieved drivers and laps:
  HAM - Lap 2
  HAM - Lap 14
  HAM - Lap 15
  HAM - Lap 17
  HAM - Lap 11


In [32]:
# ---- METADATA FILTERING FOR PRECISE QUERIES ----
# Instead of pure semantic search, filter by lap number directly
# This handles questions with numerical/positional logic

results_filtered = collection.query(
    query_embeddings=[embed_text("Hamilton tyre compound")],
    n_results=5,
    where={
        "$and": [
            {"driver": {"$eq": "HAM"}},
            {"lap": {"$gt": 26}}  # Second half of a 52-lap race
        ]
    }
)

print("Filtered results — Hamilton, laps after 26:")
for m in results_filtered['metadatas'][0]:
    print(f"  {m['driver']} - Lap {m['lap']} - {m['compound']}")

Filtered results — Hamilton, laps after 26:
  HAM - Lap 29 - MEDIUM
  HAM - Lap 47 - SOFT
  HAM - Lap 28 - MEDIUM
  HAM - Lap 31 - MEDIUM
  HAM - Lap 35 - SOFT


In [13]:
question2 = "Was Hamilton's pit stop strategy successful in terms of final position?"
answer2   = ask_easyf1(question2)
print("QUESTION 2:", question2)
print(answer2)
print("="*60)

Server busy — waiting 30 seconds (attempt 1/3)...
Server busy — waiting 30 seconds (attempt 2/3)...
QUESTION 2: Was Hamilton's pit stop strategy successful in terms of final position?
Based on the provided data, it is not possible to determine the success of Hamilton's pit stop strategy in terms of his final position. The data only shows his position at various points during the race (3rd, 5th, and 7th) and does not provide information about his final finishing position. Therefore, I cannot answer whether his strategy was successful in achieving a better final result.


In [ ]:
# Test 3 SC period timing
q3 = "How long was the safety car period and which lap did it start?"
print("QUESTION 3:")
print(ask_easyf1(q3))
print("="*60)

QUESTION 3:
The safety car period started before lap 33. Data shows Max Verstappen on SOFT tyres on lap 35, and previously on MEDIUM tyres (tyre age 33 laps) on lap 33. Verstappen pitted at the end of lap 33. The safety car was still deployed on lap 38.


In [ ]:
# Test 4 Tyre age analysis  
q4 = "How old were Hamilton's tyres when the safety car was deployed?"
print("QUESTION 4:")
print(ask_easyf1(q4))
print("="*60)

QUESTION 4:
Server busy — waiting 30 seconds (attempt 1/3)...
The provided data shows Lewis Hamilton's tyre age at different laps during the 2023 British Grand Prix while the safety car was deployed. At lap 37, his tyres were 7 laps old. Before that, at lap 35, they were 5 laps old. Two laps later, at lap 38, his tyres were 8 laps old. The data does not specify the exact lap the safety car was deployed, only that it was present during these laps.


In [16]:
# Test 5 — Verstappen (tests cross-driver retrieval)
q5 = "What was Verstappen's tyre strategy during the safety car period?"
print("QUESTION 5:")
print(ask_easyf1(q5))

QUESTION 5:
During the safety car period at the 2023 British Grand Prix, Max Verstappen ran on SOFT tyres for multiple laps. His tyre age on SOFTs progressed from 5 laps on lap 35 to 8 laps on lap 38. Before the safety car, he had completed a long stint of 33 laps on MEDIUM tyres in stint 1 before pitting.


In [13]:
questions = [
    "What tyre compound did Hamilton use in the second half of the race?",
    "Was Hamilton's pit stop strategy successful in terms of final position?",
    "How long was the safety car period and which lap did it start?",
    "How old were Hamilton's tyres when the safety car was deployed?",
    "What was Verstappen's tyre strategy during the safety car period?"
]

for i, q in enumerate(questions):
    print(f"QUESTION {i+1}: {q}")
    print(ask_easyf1(q))
    print("="*60)

QUESTION 1: What tyre compound did Hamilton use in the second half of the race?
Auto-filter: HAM detected
Auto-filter: lap range detected
Lewis Hamilton used SOFT tires in the second half of the 2023 British Grand Prix. Data shows him on SOFT tires at lap 42 with 12 laps on them, lap 48 with 18 laps on them, and lap 50 with 20 laps on them. He also used SOFT tires on the final lap at lap 52 with 22 laps on them.
QUESTION 2: Was Hamilton's pit stop strategy successful in terms of final position?
Auto-filter: HAM detected
The provided data shows Lewis Hamilton's progress during the 2023 British Grand Prix. At lap 33, he was in position 3 on worn MEDIUM tires (33 laps old). After pitting for SOFT tires, he rejoined in position 3 and remained in position 3 on lap 35. Therefore, based on this data, his pit stop strategy did not result in an improved final position up to lap 35.
QUESTION 3: How long was the safety car period and which lap did it start?
The provided data does not contain enou